In the previous post, I [read a payment transaction](https://arjunrajkumar.github.io/deconstructing-a-payment-transaction/) byte by byte. Here I want to verify the signature on its first input.

This is input 1 of 11 from the earlier post. It spends output index `0` of a previous transaction.. The input carries an **unlocking script** (`scriptSig`) with the spender’s signature and public key.

```text
Input 1 of 11 (byte offsets 5 through 184, counting from 0)
  Previous TXID : 3fd4cb75e272a5e68524d15df2d16a7b926d4998c2f1bc60a6f76083eeaba404
  Output index  : 0  (bytes: 00 00 00 00)
  Script length : 139 bytes  (encoded as: 8b)
  Unlocking script bytes:
    48 30 45 02 21 00 c6 d5 92 90 df 93 4b ac 08 bc
    41 24 b0 61 54 ee 21 39 4d 37 ec 95 8d 14 51 29
    a8 85 db d6 3e ab 02 20 03 62 51 12 0e 4c ea cc
    8e 9f f1 35 85 04 99 45 c7 db db 85 92 1a 6e 07
    7a 80 c7 e4 10 93 1f 4a 01 41 04 f3 ce 89 48 c5
    92 69 0c b3 6a 7e 6e 31 53 49 27 56 9f 41 e4 5c
    77 fb db 95 fd b0 e2 35 71 7a 29 5b 2a 28 1c 04
    61 91 39 c4 53 dd 91 39 e9 51 08 63 25 b4 02 1b
    d2 8d a6 fc 3d a8 69 43 cd 04 cf
  Sequence      : 4294967295  (bytes: ff ff ff ff)
```

Apart from this input, I also need the output that this input spends. And in that output, I need to look at the locking script `scriptPubKey` will tell me the spending conditions that the spender must match to spend the funds.

So, to summarize the current inputs unlocking script should match the previous transactions locking script.

Because I'm running a pruned node, I used a block explorer to find the block that contains the previous transaction, and then requested the block from a connected peer via `getblockfrompeer`. Once the block was available, I used `getrawtransaction` to look up the previous transaction using its ID and the block hash. I saved that reply in `previous_tx` below.

In [30]:
previous_txid = "3fd4cb75e272a5e68524d15df2d16a7b926d4998c2f1bc60a6f76083eeaba404"
previous_tx_block_hash = "00000000000007fe3e6060ec0c6ddb6790c77a8dc06eea4575a2408eebbf07d0"
previous_output_index = 0

unlocking_script_bytes = bytes.fromhex("""
    48 30 45 02 21 00 c6 d5 92 90 df 93 4b ac 08 bc
    41 24 b0 61 54 ee 21 39 4d 37 ec 95 8d 14 51 29
    a8 85 db d6 3e ab 02 20 03 62 51 12 0e 4c ea cc
    8e 9f f1 35 85 04 99 45 c7 db db 85 92 1a 6e 07
    7a 80 c7 e4 10 93 1f 4a 01 41 04 f3 ce 89 48 c5
    92 69 0c b3 6a 7e 6e 31 53 49 27 56 9f 41 e4 5c
    77 fb db 95 fd b0 e2 35 71 7a 29 5b 2a 28 1c 04
    61 91 39 c4 53 dd 91 39 e9 51 08 63 25 b4 02 1b
    d2 8d a6 fc 3d a8 69 43 cd 04 cf
""")
print(f"Unlocking script: {len(unlocking_script_bytes)} bytes")

# I got previous_tx via bitcoin cli
# !bitcoin-cli getblockfrompeer {previous_tx_block_hash} 0
# !bitcoin-cli getrawtransaction {previous_txid} true {previous_tx_block_hash}
previous_tx = {
  "in_active_chain": True,
  "txid": "3fd4cb75e272a5e68524d15df2d16a7b926d4998c2f1bc60a6f76083eeaba404",
  "hash": "3fd4cb75e272a5e68524d15df2d16a7b926d4998c2f1bc60a6f76083eeaba404",
  "version": 1,
  "size": 225,
  "vsize": 225,
  "weight": 900,
  "locktime": 0,
  "vin": [
    {
      "txid": "4e598fea2110bc7f7b1c433db4f9a489ff1a2ba68874000fe9ac8da186c318b7",
      "vout": 1,
      "scriptSig": {
        "asm": "3046022100dbff78061770bf76984c4ea7de28124b3443b96247448d9138710bbb81572cef022100af268d05fbffa315c51811c4c6baf747b6083d62eaaf12762530997c7250700b[ALL] 04337d915c620beac43c8d358158f777ea4d040099e3deb67b63a8e61361ff4f43847b05b5ee971fa3aeb748d3e553dce24b0978831984d9a0b1dbdb648438c34c",
        "hex": "493046022100dbff78061770bf76984c4ea7de28124b3443b96247448d9138710bbb81572cef022100af268d05fbffa315c51811c4c6baf747b6083d62eaaf12762530997c7250700b014104337d915c620beac43c8d358158f777ea4d040099e3deb67b63a8e61361ff4f43847b05b5ee971fa3aeb748d3e553dce24b0978831984d9a0b1dbdb648438c34c"
      },
      "sequence": 4294967295
    }
  ],
  "vout": [
    {
      "value": 50000.00000000,
      "n": 0,
      "scriptPubKey": {
        "asm": "OP_DUP OP_HASH160 fef5cca40eaf93988c09ee70728e139f65f009ff OP_EQUALVERIFY OP_CHECKSIG",
        "desc": "addr(1QF76AaXFWmMqeiKsRQvQNnz4AhJ91tkwB)#7tjw2xwp",
        "hex": "76a914fef5cca40eaf93988c09ee70728e139f65f009ff88ac",
        "address": "1QF76AaXFWmMqeiKsRQvQNnz4AhJ91tkwB",
        "type": "pubkeyhash"
      }
    }
  ],
  "hex": "0100000001b718c386a18dace90f007488a62b1aff89a4f9b43d431c7b7fbc1021ea8f594e010000008c493046022100dbff78061770bf76984c4ea7de28124b3443b96247448d9138710bbb81572cef022100af268d05fbffa315c51811c4c6baf747b6083d62eaaf12762530997c7250700b014104337d915c620beac43c8d358158f777ea4d040099e3deb67b63a8e61361ff4f43847b05b5ee971fa3aeb748d3e553dce24b0978831984d9a0b1dbdb648438c34cffffffff01005039278c0400001976a914fef5cca40eaf93988c09ee70728e139f65f009ff88ac00000000",
  "blockhash": "00000000000007fe3e6060ec0c6ddb6790c77a8dc06eea4575a2408eebbf07d0",
  "confirmations": 720551,
  "time": 1315755730,
  "blocktime": 1315755730
}

previous_output = previous_tx["vout"][previous_output_index]
previous_output_value_btc = previous_output["value"]
locking_script_hex = previous_output["scriptPubKey"]["hex"]
locking_script_asm = previous_output["scriptPubKey"]["asm"]
locking_script_type = previous_output["scriptPubKey"]["type"]

print(f"previous_output: {previous_output}")
print(f"previous_output_value_btc: {previous_output_value_btc}")
print()
print(f"locking_script_hex (hex): {locking_script_hex}")
print(f"locking_script_asm: {locking_script_asm}")
print(f"locking_script_type: {locking_script_type}")

Unlocking script: 139 bytes
previous_output: {'value': 50000.0, 'n': 0, 'scriptPubKey': {'asm': 'OP_DUP OP_HASH160 fef5cca40eaf93988c09ee70728e139f65f009ff OP_EQUALVERIFY OP_CHECKSIG', 'desc': 'addr(1QF76AaXFWmMqeiKsRQvQNnz4AhJ91tkwB)#7tjw2xwp', 'hex': '76a914fef5cca40eaf93988c09ee70728e139f65f009ff88ac', 'address': '1QF76AaXFWmMqeiKsRQvQNnz4AhJ91tkwB', 'type': 'pubkeyhash'}}
previous_output_value_btc: 50000.0

locking_script_hex (hex): 76a914fef5cca40eaf93988c09ee70728e139f65f009ff88ac
locking_script_asm: OP_DUP OP_HASH160 fef5cca40eaf93988c09ee70728e139f65f009ff OP_EQUALVERIFY OP_CHECKSIG
locking_script_type: pubkeyhash


The `locking_script_type` says that it is of type `pubkeyhash`, and the `locking_script_asm` shows the fields that go into the public key script. We can also decode `locking_script_hex` and it should match the result shown in `locking_script_asm`

## Decode the P2PKH locking script

Here are the 25 bytes of `locking_script_hex` from output `0` of the previous transaction:

```text
76 a9 14   fef5cca40eaf93988c09ee70728e139f65f009ff   88 ac
           └────── 20-byte public key hash ───────┘
```

Read the bytes from left to right. An *opcode* is a byte that tells Bitcoin which operation to perform.

| Hex | Instruction | What it does here |
| --- | --- | --- |
| `76` | `OP_DUP` | Copy the public key supplied by the spending input. |
| `a9` | `OP_HASH160` | Hash the copy with SHA-256, then RIPEMD-160. |
| `14` + the next 20 bytes | Push the expected hash | `14` in hex is 20 in decimal. Read the next 20 bytes as one value. |
| `88` | `OP_EQUALVERIFY` | Stop if the two public key hashes differ. |
| `ac` | `OP_CHECKSIG` | Use the public key to check the signature for this input. |

Bitcoin Core calls this pattern *pay to public key hash* (P2PKH). Its [P2PKH check](https://github.com/bitcoin/bitcoin/blob/v24.0.1/src/script/standard.cpp#L76-L83) fixes the length and the bytes on either side of the hash. Its [opcode definitions](https://github.com/bitcoin/bitcoin/blob/v24.0.1/src/script/script.h#L121-L186) give the names in the table.

In the readable form (`asm`), Bitcoin Core shows the pushed hash itself. The `14` has already told us how many bytes to read. The cell below applies that rule and compares the result with `locking_script_asm`. It decodes the script; checking the signature is a later step.

In [31]:
script_bytes = bytes.fromhex(locking_script_hex)

# Check the standard P2PKH pattern before reading the hash.
assert len(script_bytes) == 25
assert script_bytes[:3] == bytes.fromhex("76 a9 14")
assert script_bytes[-2:] == bytes.fromhex("88 ac")

public_key_hash = script_bytes[3:23]
decoded_asm = f"OP_DUP OP_HASH160 {public_key_hash.hex()} OP_EQUALVERIFY OP_CHECKSIG"

assert decoded_asm == locking_script_asm
print(decoded_asm)

OP_DUP OP_HASH160 fef5cca40eaf93988c09ee70728e139f65f009ff OP_EQUALVERIFY OP_CHECKSIG
